# Exercise 03 — SOLUTION: Noisy LIF with E/I Populations

Try the stub first! Only open this after attempting [ex03_stub.ipynb](ex03_stub.ipynb).

---

In [ ]:
!nvidia-smi

In [ ]:
%%writefile lif_noisy_sol.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>
#include <curand_kernel.h>

#define CUDA_CHECK(call) do { cudaError_t e=(call); if(e!=cudaSuccess){ \
    fprintf(stderr,"CUDA: %s\n",cudaGetErrorString(e));exit(1);}} while(0)

__constant__ float c_dt, c_tau_m, c_E_L, c_Rm, c_V_th, c_V_reset, c_sigma, c_I_stim;
__constant__ int   c_T_ref;

// SOLUTION: init_rng
__global__ void init_rng(curandState* states, int N, unsigned long seed) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;
    curand_init(seed, i, 0, &states[i]);  // unique sequence per neuron
}

// SOLUTION: noisy LIF step
__global__ void lif_noisy_step(
    float* V, const float* I_bias, int* ref,
    int* sid, float* st, int* ns, int max_sp,
    curandState* rng, int N, float t_ms, int stim
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;
    if (ref[i] > 0) { ref[i]--; V[i] = c_V_reset; return; }

    float I_total = I_bias[i];
    if (stim) I_total += c_I_stim;               // stimulus
    float noise = c_sigma * curand_normal(&rng[i]) * sqrtf(c_dt);  // Gaussian noise

    float dV = c_dt / c_tau_m * (-(V[i] - c_E_L) + c_Rm * I_total) + noise;
    V[i] += dV;

    if (V[i] >= c_V_th) {
        V[i] = c_V_reset; ref[i] = c_T_ref;
        int idx = atomicAdd(ns, 1);
        if (idx < max_sp) { sid[idx]=i; st[idx]=t_ms; }
    }
}

// SOLUTION: E/I initialization
__global__ void init_state_ei(float* V, float* I_bias, int* ref, int N, float I_E, float I_I) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;
    V[i]=c_E_L; ref[i]=0;
    I_bias[i] = (i < (int)(0.8f*N)) ? I_E : I_I;  // 80% E, 20% I
}

// Challenge: OU noise process kernel (bonus)
__global__ void update_ou_noise(
    float* xi, curandState* rng, int N, float tau_noise, float sigma_ou
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;
    float dxi = -c_dt / tau_noise * xi[i]
              + sigma_ou * sqrtf(2.0f * c_dt / tau_noise) * curand_normal(&rng[i]);
    xi[i] += dxi;
}

int main() {
    const int N=10000; const float T_ms=1000.f, dt=0.1f;
    const float sigma=0.5f, I_stim=1.0f, t_on=200.f, t_off=600.f;
    const float I_E=1.8f, I_I=1.2f;  // inhibitory neurons fire slower
    int T_steps=(int)(T_ms/dt);

    float tau_m=20.f,E_L=-65.f,Rm=10.f,V_th=-55.f,V_reset=-70.f;
    int T_ref=(int)(2.f/dt);
    CUDA_CHECK(cudaMemcpyToSymbol(c_dt,&dt,sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_tau_m,&tau_m,sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_E_L,&E_L,sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_Rm,&Rm,sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_V_th,&V_th,sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_V_reset,&V_reset,sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_T_ref,&T_ref,sizeof(int)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_sigma,&sigma,sizeof(float)));
    CUDA_CHECK(cudaMemcpyToSymbol(c_I_stim,&I_stim,sizeof(float)));

    float *d_V,*d_I; int *d_ref,*d_sid,*d_ns; float *d_st;
    curandState* d_rng;
    int ms=N*300;
    CUDA_CHECK(cudaMalloc(&d_V,N*4)); CUDA_CHECK(cudaMalloc(&d_I,N*4));
    CUDA_CHECK(cudaMalloc(&d_ref,N*4)); CUDA_CHECK(cudaMalloc(&d_sid,ms*4));
    CUDA_CHECK(cudaMalloc(&d_st,ms*4)); CUDA_CHECK(cudaMalloc(&d_ns,4));
    CUDA_CHECK(cudaMalloc(&d_rng,N*sizeof(curandState)));
    CUDA_CHECK(cudaMemset(d_ns,0,4));

    int thr=256,blk=(N+thr-1)/thr;
    init_rng<<<blk,thr>>>(d_rng,N,12345UL);
    init_state_ei<<<blk,thr>>>(d_V,d_I,d_ref,N,I_E,I_I);
    CUDA_CHECK(cudaDeviceSynchronize());

    cudaEvent_t t0,t1;
    CUDA_CHECK(cudaEventCreate(&t0)); CUDA_CHECK(cudaEventCreate(&t1));
    CUDA_CHECK(cudaEventRecord(t0));
    for (int step=0;step<T_steps;step++) {
        float t_now=step*dt;
        int stim=(t_now>=t_on&&t_now<t_off)?1:0;
        lif_noisy_step<<<blk,thr>>>(d_V,d_I,d_ref,d_sid,d_st,d_ns,ms,d_rng,N,t_now,stim);
    }
    CUDA_CHECK(cudaEventRecord(t1)); CUDA_CHECK(cudaEventSynchronize(t1));
    float gpu_ms; CUDA_CHECK(cudaEventElapsedTime(&gpu_ms,t0,t1));

    int h_ns; CUDA_CHECK(cudaMemcpy(&h_ns,d_ns,4,cudaMemcpyDeviceToHost));
    h_ns=(h_ns<ms)?h_ns:ms;
    int* h_sid=(int*)malloc(h_ns*4); float* h_st=(float*)malloc(h_ns*4);
    CUDA_CHECK(cudaMemcpy(h_sid,d_sid,h_ns*4,cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(h_st,d_st,h_ns*4,cudaMemcpyDeviceToHost));

    FILE* f=fopen("spikes_noisy.txt","w");
    for(int k=0;k<h_ns;k++) fprintf(f,"%d %.2f\n",h_sid[k],h_st[k]);
    fclose(f);
    printf("GPU=%.1f ms  spikes=%d  mean_fr=%.1f Hz\n",
           gpu_ms, h_ns, (float)h_ns/N/(T_ms/1000.f));
    free(h_sid); free(h_st);
    return 0;
}

In [ ]:
!nvcc -O2 -o lif_noisy_sol lif_noisy_sol.cu -lm -lcurand && ./lif_noisy_sol

In [ ]:
# Visualization (same as stub target)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
N=10000
spikes=pd.read_csv('spikes_noisy.txt',sep=' ',names=['neuron','time_ms'])
print(f"Total spikes: {len(spikes):,}  Mean FR: {len(spikes)/N/1.0:.1f} Hz")

fig,(ax1,ax2)=plt.subplots(2,1,figsize=(14,7),sharex=True,gridspec_kw={'height_ratios':[3,1]})
rng=np.random.default_rng(0)
for pop,c,n_pop,label in [('E','steelblue',int(0.8*N),'Excitatory'),
                            ('I','tomato',int(0.2*N),'Inhibitory')]:
    lo=0 if pop=='E' else int(0.8*N)
    hi=int(0.8*N) if pop=='E' else N
    ids=rng.choice(range(lo,hi),size=min(150 if pop=='E' else 50, hi-lo),replace=False)
    s=spikes[spikes['neuron'].isin(ids)]
    ax1.scatter(s['time_ms'],s['neuron'],s=0.3,c=c,alpha=0.7,label=label)
ax1.axvspan(200,600,alpha=0.1,color='yellow',label='Stimulus')
ax1.legend(fontsize=10,markerscale=8); ax1.set_ylabel('Neuron ID',fontsize=12)
ax1.set_title('Noisy LIF — E/I populations + stimulus',fontsize=13)

bins=np.arange(0,1001,10)
c1,_=np.histogram(spikes[spikes['neuron']<int(0.8*N)]['time_ms'],bins=bins)
c2,_=np.histogram(spikes[spikes['neuron']>=int(0.8*N)]['time_ms'],bins=bins)
ax2.plot(bins[:-1]+5,c1/int(0.8*N)/0.01,'b',lw=1.2,label='E')
ax2.plot(bins[:-1]+5,c2/int(0.2*N)/0.01,'r',lw=1.2,label='I')
ax2.axvspan(200,600,alpha=0.1,color='yellow'); ax2.legend(fontsize=10)
ax2.set_xlabel('Time (ms)',fontsize=12); ax2.set_ylabel('Rate (Hz)',fontsize=12)
plt.tight_layout(); plt.savefig('raster_ei_sol.png',dpi=150,bbox_inches='tight'); plt.show()